## Problem

After learning a skill like Generative AI, common questions arise:



*   What is the demand for Generative AI in the industry?
*   Which roles require this skill?
*   What jobs are available in India?



## Solution

What if you could simply ask:

"What's the demand for Generative AI skills in the industry? Show me related job openings in India"

And get current market insights, real job openings, and easy-to-apply links all in one response?

This is exactly what our SkillMap Agent will do.


In [23]:
!pip install -qU langchain
!pip install -qU langchain-google-genai
!pip install -qU langchain-tavily

In [24]:
from langchain.chat_models import init_chat_model
from google.colab import userdata

google_api_key = userdata.get('gemini_api_key')
model = init_chat_model(
    "google_genai:gemini-2.5-flash",
    api_key=google_api_key
)

In [25]:
from langchain_tavily import TavilySearch
from google.colab import userdata

tavily_api_key = userdata.get('TAVILY_API_KEY')

skill_demand_tool = TavilySearch(
  max_results=5,
  search_depth="advanced",
  tavily_api_key=tavily_api_key
)

In [26]:
from pprint import pprint
result = skill_demand_tool.invoke({"query": "generative ai skills demand 2025"})
pprint(result)

{'answer': None,
 'follow_up_questions': None,
 'images': [],
 'query': 'generative ai skills demand 2025',
 'request_id': '8c1af567-d821-42c5-a844-576d4e267d8b',
 'response_time': 0.0,
 'results': [{'content': '# Sustained Growth in AI Skill Demand\n'
                         '\n'
                         'Unique job postings for generative AI skills have '
                         'grown from 55 in January 2021 to nearly 10,000 by '
                         'May 2025. The most notable acceleration began in '
                         'early 2023, coinciding with the widespread adoption '
                         'of tools like ChatGPT, and has continued through the '
                         'present. Perhaps the clearest sign of Generative '
                         'AI’s staying power as a skill is the rise in an '
                         'associated occupation: “Generative Artificial '
                         'Intelligence Engineer,” an occupation that has seen '
                

In [27]:
system_prompt = """You are a Skill-to-Career Mapping assistant that helps students understand skill demand and find matching job opportunities.

You have access to these tools:
- skill_demand_tool: Search for industry demand, salary insights, and career trends
- search_jobs: Find actual job listings requiring specific skills

Help the student by researching the skill they ask about and finding relevant opportunities.

Present results in a clean, readable format with clear sections and proper spacing. Include all job details with apply links. Don't use markdown format."""

In [28]:
import requests
from langchain.tools import tool
from google.colab import userdata

@tool
def search_jobs(skill: str, location: str) -> list:
    """Search for jobs requiring a specific skill using JSearch API from RapidAPI."""
    print(f"\nCalling search_jobs tool")
    print(f"Searching jobs for: {skill} in {location}")

    rapidapi_key = userdata.get('RAPIDAPI_KEY')

    # Map location text to a JSearch country code instead of hardcoding "in"
    location_lower = location.lower()
    if "india" in location_lower:
        country_code = "in"
    elif "usa" in location_lower or "us" in location_lower or "united states" in location_lower:
        country_code = "us"
    else:
        country_code = "in"  # sensible default; adjust as needed

    url = "https://jsearch.p.rapidapi.com/search"
    headers = {
        "x-rapidapi-key": rapidapi_key,
        "x-rapidapi-host": "jsearch.p.rapidapi.com"
    }
    querystring = {
        "query": f"{skill} in {location}",
        "page": "1",
        "country": country_code,
        # Loosened filters so a generic skill query isn't zeroed out.
        # Add these back once you confirm results are flowing.
        # "employment_types": "INTERN,FULLTIME",
        # "job_requirements": "no_experience,under_3_years_experience"
    }

    response = requests.get(url, headers=headers, params=querystring)

    # Surface API/auth errors instead of silently returning an empty list
    if response.status_code != 200:
        print(f"JSearch API error: status {response.status_code}")
        print(response.text[:500])
        return []

    data = response.json()

    if "data" not in data:
        print(f"Unexpected JSearch response: {data}")
        return []

    jobs = data.get("data", [])
    print(f"Found {len(jobs)} jobs\n")

    result = []
    for job in jobs:
        result.append({
            "title": job.get("job_title"),
            "company": job.get("employer_name"),
            "location": job.get("job_city"),
            "apply_link": job.get("job_apply_link")
        })
    return result

In [29]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[skill_demand_tool, search_jobs],
    system_prompt=system_prompt
)

In [30]:
user_query = "What's the demand for software developer in the industry and show me related job openings in India if there are nothing in india search in usa"

response = agent.invoke({
    "messages": [{"role": "user", "content": user_query}]
})


Calling search_jobs tool
Searching jobs for: software developer in India
Found 10 jobs



In [31]:
pprint(response["messages"][-1].content)

[{'extras': {'signature': 'Cq4HARFNMg9Z8rTCtOB7y13VgZZZ/Zy6L4L/ReIpqQx20OmAs+YyTfrI18vuxUfDM9VFR5Q+t3jYRv+fji5fs88bUOthAcmQ/fBzEhR0NHRffdchUpFvYNlXHoU2M2OCOlVxaz425TaEIdJ3c0tda3++gByPPAOgE03mKgEbUF4oYc+sncxweKpOd5Q7Yy47ZQfxHvUpTlXfZbHijbclbb0BzhhQN8W+/emGuaRuI4gWafFYcC6qjkKCAkYCKuevwX0XlpGwXLf9wO+A4ItSuKfwCTMax5PjWu11wW91suCDiqhtdyV2nZ+iSshuiD+5RXlEWH7nkTdyPN7UYCcixqt4fH1A9DqEJUiU4w2Nj4H5surR6mcznGKSFRMd9i6nUa/MxATj9KRYC8uS+K+R+D87B24nYyRLAklU2P7eLi4OUzZ0ohOYOGwYIUHjXmqmtbAuHIPnriP8NERk5P/gbg5DwkcUbR5Zhgs+vKGI9YezPwcU4xs48SmQW0sJc6FCTR0pL5b7ZwuQMOL7Pzgihk386wERtBjc1Oseb6uFn61EqdcK+zUf2BREqYIAy186/lqMcscBCTYuNIm9LzDrEClHIZHvT1vY5kUcsUBGL3QGYrEPm3HbKDGZV0U9MyxBzyDHXOq9DkcJ1FRhFJHNOpy2QBuBiReuPw5QnkaRXhVJv2ko8xgUqgNJYB94y7aZP99V0rGp9YD/M7i56di3VQ4TFfQLZLj/DQN0tdoPNkJmcU5JHUIo33jUzOvCq0/+f2NEK8UN05w786qxr8bFBUeYw08rm0sWlBblfy5k5+ftiHvMcJoRYbZOHCH5537qyeyV38TyY3P/S4OIBCBT63/gqSrZeY9+YoVjXysARY8IAZMQ3nVPsbLS3nBCEgykgKgVjWUriWlkDQ6QumgVqcrg+jTDhCh3wov8VY2JbALzJG7u13NX2TYVwT+l/n3DWcDzwDXYhth31